In [1]:
from embedder import Embedder

embed = Embedder()

2026-06-28 07:53:21.357068586 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [ ]:
q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

In [13]:
v1[0]

np.float64(-0.02058203437252893)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [25]:
for doc in documents:
    if doc['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        d = doc
        break

In [27]:
d = d['content']

In [28]:
d

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [29]:
d1 = embed.encode(d)

In [30]:
d1.shape

(384,)

In [3]:
import numpy as np

In [33]:
np.dot(d1,v1)

np.float64(0.36107027225589694)

In [4]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [6]:
len(chunks)

295

In [5]:
texts = []

for chunk in chunks:
    text = chunk["content"]
    texts.append(text)

In [6]:
from tqdm.auto import tqdm

In [7]:
vectors = []

for i in tqdm(range(len(texts))):
    batch = texts[i]
    batch_vectors = embed.encode(batch)
    vectors.append(batch_vectors)
len(vectors)

  0%|          | 0/295 [00:00<?, ?it/s]

295

In [8]:
X = np.array(vectors)

In [12]:
X.shape

(295, 384)

In [10]:
q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)

In [12]:
scores = X.dot(v)

In [16]:
scores[0]

np.float64(0.3151876106372894)

In [26]:
chunks[int(np.argmax(scores))], scores[int(np.argmax(scores))]

({'start': 1000,
  'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, on

In [27]:
q = "What metric do we use to evaluate a search engine?"
v = embed.encode(q)

In [29]:
len(chunks)

295

In [30]:
scores.shape

(295,)

In [9]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [32]:
results = vindex.search(v, num_results=5)

In [33]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [10]:
# Min Search

from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

In [11]:
question = "How do I store vectors in PostgreSQL?"

search_results = index.search(
    question,
    num_results=5
)

for result in search_results:
    print(result['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [12]:
# Vector Search

q = "How do I store vectors in PostgreSQL?"
v = embed.encode(q)

In [13]:
results = vindex.search(v, num_results=5)

In [14]:
for result in results:
    print(result['filename'])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [15]:
for result in search_results:
    print(result['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [25]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked]

In [26]:
query = "How do I give the model access to tools?"
v = embed.encode(query)

In [27]:
vector_results = vindex.search(v, num_results=5)

In [28]:
text_results = index.search(
    question,
    num_results=5
)

In [29]:
results = rrf([vector_results, text_results])

In [30]:
for result in results:
    print(result['filename'])

01-agentic-rag/lessons/01-intro.md
02-vector-search/lessons/02-embeddings.md
04-evaluation/lessons/02-ground-truth.md
03-orchestration/lessons/05-rag.md
01-agentic-rag/lessons/16-other-frameworks.md
02-vector-search/lessons/01-intro.md
01-agentic-rag/lessons/15-frameworks.md
03-orchestration/lessons/05-rag.md
01-agentic-rag/lessons/13-function-calling.md
02-vector-search/lessons/01-intro.md
